# Module 07: OpenLineage — Producer & Consumer

## What You'll Learn

- What OpenLineage is and why lineage matters
- How Feast **emits** OpenLineage events (producer role)
- How Feast **receives and stores** OpenLineage events from other tools (consumer role — **new in 2026**)
- Configuring both producer and consumer in `feature_store.yaml`
- The Feast consumer API: ingesting events, querying the lineage graph
- Consuming lineage from Spark, Airflow, and dbt **without Marquez**
- What the consumer covers, what it doesn't, and when to still use Marquez

---

> **🗺️ DATA STRATEGY**: This is **Workshop Decision #1** — E2E Lineage via OpenLineage. Full data audit trail: source → transformations → model → agent. Feast was the first production OpenLineage **emitter** in the RHOAI stack. As of June 2026 it also functions as a native OpenLineage **consumer** — teams can receive cross-producer lineage events without deploying Marquez separately. This is tracked as RHAISTRAT-2335 (Tech Preview, RHOAI 3.6).

> **📦 VERSION NOTE**: Consumer support landed in feast upstream via [PR #6549](https://github.com/feast-dev/feast/pull/6549) (Jun 25 2026), enhanced in [PR #6719](https://github.com/feast-dev/feast/pull/6719) (Aug 18) and [PR #6759](https://github.com/feast-dev/feast/pull/6759) (Aug 20). It is present in [opendatahub-io/feast](https://github.com/opendatahub-io/feast) midstream.

## What Is OpenLineage?

**OpenLineage** is an open standard (LFAI graduate project) for collecting and analysing lineage metadata. It defines a common **event schema** that any tool can emit, and a simple **HTTP transport** that any server can receive.

### Core Concepts

| Concept | Meaning |
|---------|---------|
| **RunEvent** | A job execution: start, complete, or fail. Has inputs and outputs. |
| **DatasetEvent** | A standalone dataset metadata update (schema change, etc.) |
| **JobEvent** | A standalone job metadata update |
| **Facet** | A typed, versioned extension on a RunEvent, Dataset, or Job (e.g. `SchemaDatasetFacet`, `FeastFeatureViewFacet`) |
| **Namespace** | Logical grouping for jobs and datasets (e.g. a project or cluster) |
| **Producer** | Any tool that *emits* OpenLineage events (Spark, Airflow, dbt, Feast) |
| **Consumer** | A server that *receives, stores, and queries* events (Marquez, DataHub, **now Feast itself**) |

### The Architecture — Before vs After

**Before (Feast 2025 and earlier):**

```
Producers (emit events):          Consumer (stores & visualizes):
┌──────────────────────┐       ┌────────────────┐
│ Feast apply/materialize│─────▶│                │
└──────────────────────┘       │    Marquez     │
┌──────────────────────┐       │  (separate     │
│ Spark (jobs)         │─────▶│   deployment)  │
└──────────────────────┘       │                │
┌──────────────────────┐       │                │
│ Airflow (pipelines)  │─────▶│                │
└──────────────────────┘       └────────────────┘
```

**After (Feast 2026 — consumer built in):**

```
OpenLineage Producers:            Feast (producer + consumer):
┌──────────────────────┐       ┌────────────────────────────┐
│ Feast apply/materialize│─────▶│ POST /api/v1/lineage        │
└──────────────────────┘  ┌───▶│ (auto-ingests its own       │
┌──────────────────────┐  │    │  events without HTTP)       │
│ Spark (OL listener)  │──┤    ├────────────────────────────┤
└──────────────────────┘  │    │ SQL-backed lineage store:   │
┌──────────────────────┐  │    │  openlineage_events         │
│ Airflow (OL operator)│──┤    │  openlineage_jobs           │
└──────────────────────┘  │    │  openlineage_datasets       │
┌──────────────────────┐  │    │  openlineage_runs           │
│ dbt (OL integration) │──┘    │  openlineage_lineage_edges  │
└──────────────────────┘       ├────────────────────────────┤
                               │ Feast UI — Lineage tab:     │
                               │  Graph, jobs, datasets,     │
                               │  runs with filter/focus     │
                               └────────────────────────────┘
```

**Why this matters:**
- **No separate Marquez deployment** for teams who only need lineage within their Feast-centric pipelines
- **Unified view**: Feast objects (FeatureViews, DataSources) and external producer events in one graph
- **RBAC-integrated**: namespace mapping ties external producer namespaces to Feast project permissions
- **Compliance**: Gartner MQ explicitly asked Red Hat about lineage capabilities
- **Debugging**: "Why did my model's performance degrade?" → trace back through Spark → Feast → online store

## Part 1: Feast as a Producer — What Events Are Emitted

Feast emits OpenLineage events automatically for all major operations. No code changes required.

### Events Emitted by Operation

| Operation | Event Type | What It Records |
|-----------|-----------|----------------|
| `feast apply` — entities | `RunEvent (COMPLETE)` | Entity join keys and value types |
| `feast apply` — data sources | `RunEvent (COMPLETE)` | Source path, type, timestamp fields, field mappings |
| `feast apply` — feature views | `RunEvent (COMPLETE)` | DataSource + Entity → FeatureView dependency graph |
| `feast apply` — on-demand FVs | `RunEvent (COMPLETE)` | FeatureView(s) + RequestSource → ODFV |
| `feast apply` — feature services | `RunEvent (COMPLETE)` | FeatureViews → FeatureService |
| `feast apply` — saved datasets | `RunEvent (COMPLETE)` | FeatureService + DataSource → SavedDataset (via storage matching) |
| `feast materialize` | `RunEvent (START/COMPLETE/FAIL)` | Offline store → Online store data movement |
| Registry API/gRPC (create/update/delete) | `RunEvent (COMPLETE)` | Same as above — wired at RegistryServer layer |

### Jobs Created per `feast apply`

```
feast_apply_entities
feast_apply_data_sources
feast_apply_feature_view_{name}     inputs: [DataSource, Entity]    output: FeatureView
feast_apply_odfv_{name}             inputs: [FeatureView, RequestSource]  output: ODFV
feast_apply_feature_service_{name}  inputs: [FeatureView(s), ODFV(s)]     output: FeatureService
feast_apply_saved_dataset_{name}    inputs: [FeatureService, DataSource]  output: SavedDataset
```

### Custom Feast Facets Emitted

Each dataset in a RunEvent carries a custom Feast facet with rich metadata:

| Facet | Feast Object | Key Fields |
|-------|-------------|-----------|
| `FeastFeatureViewFacet` | FeatureView / StreamFV / ODFV | `name`, `ttl_seconds`, `entities`, `features`, `mode`, `online_enabled`, `tags` |
| `FeastFeatureServiceFacet` | FeatureService | `feature_views`, `feature_count`, `logging_enabled` |
| `FeastDataSourceFacet` | DataSource | `source_type`, `timestamp_field`, `field_mapping`, `path`/`table`/`query` |
| `FeastEntityFacet` | Entity | `join_keys`, `value_type` |
| `FeastSavedDatasetFacet` | SavedDataset | `features`, `join_keys`, `feature_service_name` |
| `FeastMaterializationFacet` | (on RunEvent, not dataset) | `feature_views`, `start_date`, `end_date`, `rows_written`, `online_store_type` |
| `FeastOnlineStoreFacet` | Online store target | `store_type`, `feature_view` |
| `FeastJobKindFacet` | (on Job) | `kind`: `definition` (apply) or `transform` (materialize) |

### Example RunEvent from `feast apply`

```json
{
  "eventType": "COMPLETE",
  "eventTime": "2026-09-08T10:30:00Z",
  "producer": "feast",
  "run": {"runId": "abc-123"},
  "job": {
    "namespace": "credit_scoring",
    "name": "feast_apply_feature_view_credit_history",
    "facets": {
      "feast_jobKind": {"kind": "definition", "feast_project": "credit_scoring"}
    }
  },
  "inputs": [
    {"namespace": "credit_scoring", "name": "credit_scoring.credit_timeseries_source",
     "facets": {"feast_dataSource": {"source_type": "FileSource", "timestamp_field": "event_timestamp"}}},
    {"namespace": "credit_scoring", "name": "credit_scoring.entity.customer",
     "facets": {"feast_entity": {"join_keys": ["customer_id"], "value_type": "INT64"}}}
  ],
  "outputs": [
    {"namespace": "credit_scoring", "name": "credit_scoring.credit_history",
     "facets": {
       "feast_featureView": {"ttl_seconds": 86400, "entities": ["customer"], "features": ["credit_score", "transaction_count"]},
       "schema": {"fields": [{"name": "credit_score", "type": "INT64"}, {"name": "transaction_count", "type": "INT64"}]}
     }}
  ]
}
```

## Part 2: Enabling OpenLineage — Configuration Reference

### Producer-Only Configuration (emit events to Marquez or other consumers)

```yaml
# feature_store.yaml
project: credit_scoring
registry:
  registry_type: sql
  path: sqlite:///data/registry.db   # SQL registry required for consumer; also fine for producer-only
provider: local
online_store:
  type: sqlite
  path: data/online.db

openlineage:
  enabled: true
  transport_type: http               # http | console | file | kafka
  transport_url: http://marquez-api:5000
  transport_endpoint: api/v1/lineage
  namespace: credit_scoring          # default: uses the Feast project name
  emit_on_apply: true
  emit_on_materialize: true
```

> **⚠️ Config change from 2025**: The old `lineage.backend_url` key is replaced by `openlineage.transport_url`. Update your `feature_store.yaml` if you were using the old format.

### Producer + Consumer Configuration (Feast receives events AND emits them)

```yaml
# feature_store.yaml
project: credit_scoring
registry:
  registry_type: sql                 # ← REQUIRED for consumer (SQLite, PostgreSQL, or MySQL)
  path: sqlite:///data/registry.db

openlineage:
  enabled: true
  transport_type: http
  transport_url: http://localhost:8888   # point producer at itself when self-hosting
  transport_endpoint: api/v1/lineage
  namespace: credit_scoring
  consumer:
    enabled: true
    store_type: sql                  # stores lineage in the same DB as the registry
    api_key: your-secret-key         # optional: require producers to authenticate
    retention_days: 30               # auto-prune events older than 30 days (0 = keep forever)
    namespace_mapping:               # map external OL namespaces to Feast projects (for RBAC)
      "spark://ml-cluster": "credit_scoring"
      "airflow://prod": "credit_scoring"
```

### Key Configuration Options

| Option | Default | Notes |
|--------|---------|-------|
| `openlineage.enabled` | `false` | Master switch |
| `openlineage.transport_type` | `None` | `http`, `console`, `file`, `kafka`. `None` defers to OL SDK env vars |
| `openlineage.transport_url` | — | Required when `transport_type: http` |
| `openlineage.namespace` | `feast` | When `feast`, uses project name as namespace |
| `consumer.enabled` | `false` | Enable the consumer (ingest + query endpoints + UI) |
| `consumer.store_type` | `sql` | Only `sql` is supported today |
| `consumer.connection_string` | — | Optional: separate DB. Default: uses registry DB |
| `consumer.api_key` | — | API key for producers sending events. None = open |
| `consumer.retention_days` | `30` | Auto-prune. `0` = disabled |
| `consumer.namespace_mapping` | `{}` | Maps OL producer namespaces → Feast project names |

### Environment Variables (alternative to feature_store.yaml)

```bash
export FEAST_OPENLINEAGE_ENABLED=true
export FEAST_OPENLINEAGE_TRANSPORT_TYPE=http
export FEAST_OPENLINEAGE_URL=http://marquez:5000
export FEAST_OPENLINEAGE_CONSUMER_ENABLED=true
export FEAST_OPENLINEAGE_CONSUMER_API_KEY=your-secret-key
export FEAST_OPENLINEAGE_CONSUMER_NAMESPACE_MAPPING='{"spark://ml-cluster":"credit_scoring"}'
```

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import yaml

# Setup feature store with OpenLineage consumer enabled
os.makedirs("data", exist_ok=True)
os.makedirs("feature_repo", exist_ok=True)

# Generate sample data
np.random.seed(42)
records = []
for cid in range(1, 51):
    for week in range(26):
        ts = datetime(2024, 1, 1) + timedelta(weeks=week)
        records.append({
            "customer_id": cid,
            "event_timestamp": ts,
            "credit_score": np.random.randint(300, 850),
            "transaction_count": np.random.randint(0, 500),
        })
pd.DataFrame(records).to_parquet("data/lineage_demo.parquet")

# Config with consumer enabled (self-hosted: Feast receives its own events)
# Using 'console' transport so events are printed to stdout for demo purposes.
# In production, set transport_type: http and transport_url: <lineage server>
config = {
    "project": "lineage_demo",
    "provider": "local",
    "registry": {
        "registry_type": "sql",
        "path": "sqlite:///data/registry.db"   # SQL registry required for consumer
    },
    "offline_store": {"type": "duckdb"},
    "online_store": {"type": "sqlite", "path": "data/online.db"},
    "entity_key_serialization_version": 3,
    "openlineage": {
        "enabled": True,
        "transport_type": "console",   # prints events to stdout for this demo
        "namespace": "lineage_demo",
        "emit_on_apply": True,
        "emit_on_materialize": True,
        "consumer": {
            "enabled": True,
            "store_type": "sql",       # uses registry DB (data/registry.db)
            "retention_days": 30,
        },
    },
}

with open("feature_repo/feature_store.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ Feature store configured with OpenLineage consumer enabled")
print("\nThe consumer will:")
print("  • Accept POST /api/v1/lineage from any OpenLineage producer")
print("  • Auto-ingest Feast's own apply/materialize events (no HTTP round-trip)")
print("  • Store events in SQLite (data/registry.db) in 7 lineage tables")
print("  • Expose a lineage graph API and Feast UI Lineage tab")

In [ ]:
from feast import Entity, FeatureView, Field, FileSource, FeatureStore
from feast.types import Int64

customer = Entity(name="customer", join_keys=["customer_id"])
source = FileSource(
    name="credit_source",
    path=os.path.abspath("data/lineage_demo.parquet"),
    timestamp_field="event_timestamp",
)
credit_fv = FeatureView(
    name="credit_history",
    entities=[customer],
    ttl=timedelta(weeks=2),
    schema=[
        Field(name="credit_score", dtype=Int64),
        Field(name="transaction_count", dtype=Int64),
    ],
    source=source,
)

store = FeatureStore(repo_path="feature_repo")

# feast apply emits OpenLineage events automatically.
# With consumer enabled, events are also auto-ingested into the local SQL store.
# With transport_type=console, the JSON events print to stdout.
store.apply([customer, source, credit_fv])
print("✅ feast apply complete")
print("\nEvents emitted and ingested:")
print("  feast_apply_entities        → Entity: customer")
print("  feast_apply_data_sources    → DataSource: credit_source")
print("  feast_apply_feature_view_credit_history")
print("    inputs:  [credit_source, entity.customer]")
print("    outputs: [credit_history] (with FeastFeatureViewFacet + SchemaDatasetFacet)")

In [ ]:
# Materialize — emits START + COMPLETE (or FAIL) RunEvents automatically
store.materialize(
    start_date=datetime(2024, 1, 1),
    end_date=datetime.now(),
)
print("✅ feast materialize complete")
print("\nEvents emitted and auto-ingested:")
print("  RunEvent (START)    job: lineage_demo.materialize.credit_history")
print("  RunEvent (COMPLETE) job: lineage_demo.materialize.credit_history")
print("    inputs:  [credit_source] (offline store read)")
print("    outputs: [online_store_credit_history] (online store write)")
print()
print("Facets on the COMPLETE event:")
print("  FeastMaterializationFacet:")
print("    feature_views: [credit_history]")
print("    start_date / end_date")
print("    rows_written: <n>")
print("    online_store_type: sqlite")
print("    offline_store_type: duckdb")

## Part 3: Feast as a Consumer — Receiving External Lineage Events

When the consumer is enabled, Feast exposes a standard OpenLineage HTTP endpoint at `POST /api/v1/lineage`. Any OpenLineage-compatible producer (Spark, Airflow, dbt, custom code) can POST events to it. Feast stores the events, extracts the lineage graph, and displays it in the UI.

### Starting the Consumer

```bash
# Start the Feast UI + registry server (includes consumer endpoints)
feast ui

# OR start a standalone lineage-only server (new in PR #6759)
feast serve_lineage --host 0.0.0.0 --port 6580
```

### Consumer API Endpoints

**Ingest (OpenLineage-spec compliant):**

| Method | Path | Description |
|--------|------|-------------|
| `POST` | `/api/v1/lineage` | Single RunEvent, DatasetEvent, or JobEvent |
| `POST` | `/api/v1/lineage/batch` | Array of events. Returns 204 (all ok) or 207 (partial failure) |

**Query (Feast-specific):**

| Method | Path | Description |
|--------|------|-------------|
| `GET` | `/api/v1/lineage/openlineage/graph` | Full graph: nodes + edges + symlinks |
| `GET` | `/api/v1/lineage/openlineage/graph/{node_type}/{ns}/{name}` | Subgraph from a node (`?depth=10&direction=both/upstream/downstream`) |
| `GET` | `/api/v1/lineage/openlineage/namespaces` | All known namespaces |
| `GET` | `/api/v1/lineage/openlineage/jobs` | All jobs |
| `GET` | `/api/v1/lineage/openlineage/datasets` | All datasets |
| `GET` | `/api/v1/lineage/openlineage/events` | Raw events (`?namespace=&job_name=&limit=&offset=`) |
| `GET` | `/api/v1/lineage/openlineage/runs` | Run history (`?job_namespace=&job_name=`) |
| `GET` | `/api/v1/lineage/openlineage/runs/{run_id}` | Single run with inputs/outputs |

**Admin (requires API key if configured):**

| Method | Path | Description |
|--------|------|-------------|
| `GET` | `/api/v1/lineage/openlineage/retention` | Retention config and storage stats |
| `POST` | `/api/v1/lineage/openlineage/retention/prune` | Manually trigger pruning |
| `DELETE` | `/api/v1/lineage/openlineage/reset` | Purge all or `?namespace=X` data |

### Authentication
- **Ingest and admin endpoints**: require `X-API-Key: <key>` or `Authorization: Bearer <key>` when `consumer.api_key` is set
- **Query endpoints**: no auth required (RBAC filtering applied when Feast authz is configured)
- When `consumer.api_key` is not set: all endpoints are open

In [ ]:
import requests
import json
import uuid
from datetime import datetime, timezone

# ─────────────────────────────────────────────────────────────────────────────
# Simulating what an external producer (Spark, Airflow, dbt) would send to
# the Feast OpenLineage consumer endpoint.
#
# In a real pipeline, the OpenLineage Spark listener or Airflow provider
# sends these events automatically. Here we construct them manually to
# demonstrate the ingestion API.
# ─────────────────────────────────────────────────────────────────────────────

FEAST_URL = "http://localhost:8888"   # Feast UI server
API_KEY = None                         # set if consumer.api_key is configured

def post_lineage_event(event: dict, api_key: str = None) -> dict:
    """POST a single OpenLineage event to the Feast consumer."""
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["X-API-Key"] = api_key
    resp = requests.post(
        f"{FEAST_URL}/api/v1/lineage",
        headers=headers,
        json=event,
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()

# ── Example 1: Spark ETL job that writes the credit data Feast reads ──────────

spark_event = {
    "eventType": "COMPLETE",
    "eventTime": datetime.now(timezone.utc).isoformat(),
    "producer": "https://github.com/OpenLineage/OpenLineage/tree/1.x/integration/spark",
    "schemaURL": "https://openlineage.io/spec/2-0-2/OpenLineage.json",
    "run": {
        "runId": str(uuid.uuid4()),
        "facets": {
            "spark_version": {"_producer": "spark", "spark-version": "3.5.0"}
        }
    },
    "job": {
        "namespace": "spark://ml-cluster",
        "name": "etl.credit_data_enrichment",
        "facets": {
            "jobType": {"jobType": "BATCH", "integration": "SPARK", "processingType": "BATCH"}
        }
    },
    "inputs": [{
        "namespace": "postgresql://db.internal:5432",
        "name": "public.raw_transactions",
        "facets": {
            "schema": {
                "fields": [
                    {"name": "customer_id", "type": "BIGINT"},
                    {"name": "txn_amount", "type": "DECIMAL"},
                    {"name": "txn_date", "type": "TIMESTAMP"},
                ]
            },
            "dataSource": {"name": "postgres-prod", "uri": "postgresql://db.internal:5432"}
        }
    }],
    "outputs": [{
        "namespace": "s3://ml-data",
        "name": "credit/lineage_demo.parquet",
        "facets": {
            "schema": {
                "fields": [
                    {"name": "customer_id", "type": "BIGINT"},
                    {"name": "event_timestamp", "type": "TIMESTAMP"},
                    {"name": "credit_score", "type": "INT"},
                    {"name": "transaction_count", "type": "INT"},
                ]
            },
            "dataSource": {"name": "s3-ml", "uri": "s3://ml-data"},
            # SymlinksDatasetFacet links this Spark output to the Feast DataSource
            # that reads the same S3 path, connecting the cross-producer graph
            "symlinks": {
                "identifiers": [{
                    "namespace": "lineage_demo",
                    "name": "lineage_demo.credit_source",
                    "type": "TABLE"
                }]
            }
        }
    }],
}

print("Simulated Spark RunEvent (would be sent by OpenLineage Spark listener):")
print(json.dumps(spark_event, indent=2)[:1200], "...")
print()
print("Key points:")
print("  • namespace 'spark://ml-cluster' → maps to Feast project 'lineage_demo' via namespace_mapping")
print("  • SymlinksDatasetFacet connects Spark output to Feast DataSource 'lineage_demo.credit_source'")
print("  • The lineage graph will show: postgres → [Spark ETL] → S3 ←symlink→ Feast DataSource → credit_history")
print()
print("# To actually send this (when Feast UI is running):")
print("# result = post_lineage_event(spark_event)")
print("# print(result)  # {'event_id': 'uuid...'}")

In [ ]:
# ── Example 2: Airflow DAG step emitting lineage ──────────────────────────────

airflow_event = {
    "eventType": "COMPLETE",
    "eventTime": datetime.now(timezone.utc).isoformat(),
    "producer": "https://github.com/OpenLineage/OpenLineage/tree/1.x/integration/airflow",
    "schemaURL": "https://openlineage.io/spec/2-0-2/OpenLineage.json",
    "run": {
        "runId": str(uuid.uuid4()),
        "facets": {
            "parent": {
                "run": {"runId": str(uuid.uuid4())},
                "job": {
                    "namespace": "airflow://prod",
                    "name": "credit_pipeline.trigger_feast_materialize"
                }
            }
        }
    },
    "job": {
        "namespace": "airflow://prod",
        "name": "credit_pipeline.validate_credit_data",
    },
    "inputs": [{
        "namespace": "s3://ml-data",
        "name": "credit/lineage_demo.parquet",
    }],
    "outputs": [{
        "namespace": "s3://ml-data",
        "name": "credit/validated_lineage_demo.parquet",
        "facets": {
            "columnLineage": {
                "fields": {
                    "credit_score": {
                        "inputFields": [
                            {"namespace": "s3://ml-data", "name": "credit/lineage_demo.parquet", "field": "credit_score"}
                        ],
                        "transformationDescription": "range clamp 300-850"
                    }
                }
            }
        }
    }],
}

print("Simulated Airflow RunEvent (would be sent by OpenLineage Airflow provider):")
print()
print("Key points:")
print("  • parentRun facet links this task to its parent DAG run")
print("  • columnLineage facet provides column-level lineage (FROM Airflow, not from Feast)")
print("  • Feast consumer stores the full facet JSON — it surfaces in the graph UI")
print("  • The consumer does NOT parse/index column-level lineage (no column-level query API)")
print()

# ── Example 3: Batch event submission ─────────────────────────────────────────

print("Batch ingestion (POST /api/v1/lineage/batch):")
print()
batch = [spark_event, airflow_event]
print(f"  Sending batch of {len(batch)} events")
print("  Response: 204 No Content (all ok) or 207 with failure summary")
print()
print("# To actually send:")
print("# resp = requests.post(f'{FEAST_URL}/api/v1/lineage/batch', json=batch)")
print("# print(resp.status_code, resp.json())")

In [ ]:
# ── Querying the Lineage Graph ─────────────────────────────────────────────────
#
# These are the query API calls you'd make from a notebook or custom dashboard.
# All endpoints available at /api/v1/lineage/openlineage/...
# No auth required for read-only endpoints.

def query_lineage(path: str) -> dict:
    """Query the Feast lineage API."""
    resp = requests.get(f"{FEAST_URL}/api/v1/lineage/openlineage/{path}", timeout=10)
    resp.raise_for_status()
    return resp.json()

print("=== Lineage Query API Examples ===")
print()

print("1. List all namespaces:")
print("   GET /api/v1/lineage/openlineage/namespaces")
print("   Response: {'namespaces': ['lineage_demo', 'spark://ml-cluster', 'airflow://prod']}")
print()

print("2. List all jobs across producers:")
print("   GET /api/v1/lineage/openlineage/jobs")
print("   Response: {'jobs': [")
print("     {'job_namespace': 'spark://ml-cluster', 'job_name': 'etl.credit_data_enrichment', 'producer': 'spark'},")
print("     {'job_namespace': 'lineage_demo', 'job_name': 'feast_apply_feature_view_credit_history', 'producer': 'feast'},")
print("   ]}")
print()

print("3. Full lineage graph (all nodes + edges + symlinks):")
print("   GET /api/v1/lineage/openlineage/graph")
print("   Response: {")
print("     'nodes': [<datasets>, <jobs>],")
print("     'edges': [{'source_type': 'dataset', 'source_name': '...', 'target_type': 'job', ...}],")
print("     'symlinks': [{'dataset_name': '...', 'linked_name': '...'}],")
print("   }")
print()

print("4. Subgraph around a specific node (e.g. credit_history feature view):")
print("   GET /api/v1/lineage/openlineage/graph/dataset/lineage_demo/lineage_demo.credit_history")
print("   ?depth=5&direction=upstream")
print("   → Returns: postgres.raw_transactions → Spark ETL → S3 → (symlink) → Feast DataSource → credit_history")
print()

print("5. Run history for a specific job:")
print("   GET /api/v1/lineage/openlineage/runs?job_namespace=lineage_demo&job_name=feast_apply_feature_view_credit_history")
print("   Response: {'runs': [{'run_id': '...', 'event_type': 'COMPLETE', 'event_time': '...'}], 'total': 1}")
print()

print("6. Single run detail (inputs + outputs):")
print("   GET /api/v1/lineage/openlineage/runs/{run_id}")
print("   Response: {'run_id': '...', 'inputs': [...], 'outputs': [...]}")

## Part 4: Configuring External Producers to Send to Feast

Once the Feast consumer is running, point any OpenLineage producer at `http://<feast-host>:8888/api/v1/lineage`.

### Spark (OpenLineage Spark listener)

```bash
# Add to spark-submit or SparkConf
--conf spark.extraJavaOptions=-javaagent:/path/to/openlineage-spark.jar
--conf spark.openlineage.transport.type=http
--conf spark.openlineage.transport.url=http://feast-server:8888
--conf spark.openlineage.transport.endpoint=api/v1/lineage
--conf spark.openlineage.transport.auth.type=api_key
--conf spark.openlineage.transport.auth.apiKey=your-consumer-api-key
--conf spark.openlineage.namespace=spark://ml-cluster
```

The Spark listener emits `RunEvent(START)` when a Spark job begins and `RunEvent(COMPLETE/FAIL)` when it ends, with input/output datasets including `SchemaDatasetFacet`, `dataSource` facets, and optionally `columnLineage` facets.

### Airflow (OpenLineage Airflow provider)

```python
# airflow.cfg or environment variables
OPENLINEAGE_URL=http://feast-server:8888
OPENLINEAGE_ENDPOINT=api/v1/lineage
OPENLINEAGE_API_KEY=your-consumer-api-key
OPENLINEAGE_NAMESPACE=airflow://prod
```

Or in `airflow.cfg`:
```ini
[openlineage]
transport = {"type": "http", "url": "http://feast-server:8888", "endpoint": "api/v1/lineage", "auth": {"type": "api_key", "apiKey": "your-key"}}
```

### dbt (OpenLineage dbt integration)

```yaml
# profiles.yml extension or env vars
OPENLINEAGE_URL=http://feast-server:8888
OPENLINEAGE_API_KEY=your-consumer-api-key

# Or use openlineage-dbt package configuration
```

### KFP (Kubeflow Pipelines)

```python
# In a KFP component, use the openlineage-python client directly
import os
os.environ["OPENLINEAGE_URL"] = "http://feast-server:8888"
os.environ["OPENLINEAGE_API_KEY"] = "your-consumer-api-key"
os.environ["OPENLINEAGE_NAMESPACE"] = "kfp://prod-cluster"

from openlineage.client import OpenLineageClient
client = OpenLineageClient()
# client now auto-sends events to Feast
```

### Connecting Producers to Feast Objects via SymlinksDatasetFacet

When a Spark job writes data that Feast reads as a `FileSource` or `BigQuerySource`, the two sides reference the same physical data but use different namespaces. Use the `SymlinksDatasetFacet` in the Spark output to explicitly link them:

```json
"symlinks": {
  "identifiers": [{
    "namespace": "<feast-project-name>",
    "name": "<feast-project>.<feast-datasource-name>",
    "type": "TABLE"
  }]
}
```

Feast also auto-links datasets that share the same `dataSource.uri` across producers, even without explicit symlinks. This happens in the processor's `_process_dataset_symlinks` method.

## Part 5: Feast Consumer vs Marquez — When to Use Each

### Feast Built-In Consumer (new in 2026)

The Feast Lineage tab in the UI visualises both Feast registry objects and external producer events in a single graph. It launches with `feast ui` — no extra deployment needed.

**What the Feast UI shows:**
- Full lineage graph with producer-based colour coding (Feast events vs Spark vs Airflow)
- Upstream/downstream directional focus — traverse without cross-branch bleed
- Feast object type filtering (show only FeatureViews, DataSources, etc.)
- Run history per job (START/COMPLETE/FAIL timeline)
- Dataset side panel with source metadata (path, table, query, field mappings, TTL)
- Auto-defaults to Feast-only registry view when no external OL events exist yet

**What the Feast consumer does NOT provide compared to Marquez:**

| Capability | Feast Consumer | Marquez |
|-----------|---------------|---------|
| Dataset versioning | ❌ No versioning model | ✅ Full version history per dataset |
| Column-level lineage queries | ❌ Stores facet JSON but no index | ✅ First-class column-level graph |
| Point-in-time queries | ❌ Current state only | ✅ "What did the graph look like on date X?" |
| OpenLineage Marquez API compatibility | ❌ Feast has its own query API | ✅ Marquez-compatible REST API (OpenAPI spec) |
| Dataset deprecation / lifecycle states | ❌ No lifecycle model | ✅ Active/deprecated/deleted states |
| Namespace management UI | ❌ Namespaces from events only | ✅ Explicit namespace management |
| Standalone web UI (not embedded) | ❌ Embedded in Feast UI | ✅ Standalone SPA |
| Multi-tenant isolation beyond RBAC | ❌ Namespace mapping only | ✅ Full namespace-level isolation |

### When to Use Feast Consumer

- Your lineage chain is primarily Feast-centric (Spark/Airflow feeding Feast → model training)
- You want unified visibility without adding infrastructure
- You're on RHOAI and want RBAC-integrated lineage (namespace_mapping ties to Feast project permissions)
- You're doing a TP evaluation or early adoption (RHOAI 3.6)

### When to Still Use Marquez

- You need **dataset versioning** — tracking what the schema looked like at each run
- You need **column-level lineage queries** (not just storage)
- You need **point-in-time graph reconstruction**
- You have a large multi-team deployment with producers that are NOT Feast (Spark-only pipelines)
- You need the **Marquez REST API** for integration with other tools (DataHub federation, etc.)
- You need a **standalone UI** that can be independently deployed and accessed

### Deploying Marquez (still valid alongside Feast consumer)

Marquez and the Feast consumer can coexist: configure the Feast producer to emit to both. Just point `openlineage.transport_url` at Marquez, and the Feast consumer self-ingests its own events locally.

```bash
# Deploy Marquez (if needed for advanced features)
oc new-project lineage-system
oc apply -f manifests/marquez/
oc get routes -n lineage-system | grep marquez-web
```

> **⚠️ Marquez project health note**: v0.50.0 is the latest release (Oct 2024, 19-month gap as of Sep 2026). Zero built-in authentication. Single-maintainer concentration (58% of commits). The Feast consumer addresses the most critical gap (self-hosted lineage without Marquez) while Marquez remains the reference for the full OpenLineage query spec.

## Part 6: Lineage Coverage — What's Supported and What's Not

### What Feast Now Covers (Producer + Consumer)

| Capability | Supported | Notes |
|-----------|-----------|-------|
| `feast apply` lineage — FeatureView | ✅ | DataSource + Entity → FeatureView |
| `feast apply` lineage — ODFV | ✅ | FeatureView + RequestSource → ODFV |
| `feast apply` lineage — FeatureService | ✅ | FeatureViews → FeatureService |
| `feast apply` lineage — SavedDataset | ✅ | FeatureService + DataSource (storage match) |
| `feast apply` lineage — Entities | ✅ | Added in PR #6719 |
| `feast apply` lineage — DataSources | ✅ | Added in PR #6719 |
| `feast materialize` lineage | ✅ | START/COMPLETE/FAIL |
| Registry API/gRPC changes emit lineage | ✅ | Added in PR #6719 (not just CLI apply) |
| Receive events from Spark / Airflow / dbt | ✅ | Standard OpenLineage HTTP ingest |
| Cross-producer graph traversal | ✅ | Upstream/downstream, configurable depth |
| SymlinksDatasetFacet cross-linking | ✅ | Auto-links datasets sharing a `dataSource` URI |
| Run history per job | ✅ | With input/output datasets |
| Feast UI Lineage tab | ✅ | Graph, jobs, datasets, run history |
| Standalone lineage server (`feast serve_lineage`) | ✅ | Independent scaling from registry/UI |
| Operator CRD lineage server deployment | ✅ | `lineageServer` under `openlineage.consumer` |
| RBAC namespace filtering | ✅ | `namespace_mapping` ties to Feast project permissions |
| Retention/pruning | ✅ | Configurable `retention_days` with background task |
| `consumer.api_key` authentication | ✅ | X-API-Key or Bearer token |

### What's NOT Covered (Remaining Gaps)

| Gap | Why It's a Gap | Workaround / Status |
|-----|---------------|---------------------|
| **Column-level lineage queries** | The consumer stores `columnLineage` facets but doesn't index them — no column-level query API | The raw facet JSON is queryable via `/events` endpoint; Marquez for structured column queries |
| **Dataset versioning** | No versioning model — the store upserts datasets, there's no history per dataset state | Marquez has full version history; Feast only tracks current state |
| **Point-in-time graph queries** | No "what did the graph look like on date X?" | Marquez capability; Feast shows current accumulated state |
| **Model training lineage** | MLflow does not emit OpenLineage events yet | MLflow upstream issue still open; workaround: manual OL client in training code |
| **Inference/serving lineage** | No runtime lineage for feature serving | Future RFC; model serving teams must instrument manually |
| **Feature retrieval lineage** | `get_online_features` / `get_historical_features` not tracked | `FeastRetrievalFacet` exists in the spec but is not auto-emitted per query |
| **Ray native integration** | Ray has no OpenLineage listener | Manual instrumentation with openlineage-python client |
| **File-based registry support** | Consumer requires `registry_type: sql` | No SQLite file registry; must migrate to SQL registry |
| **Marquez API compatibility** | Feast uses its own query API (not Marquez's) | Tools expecting Marquez's `/v1/namespaces`, `/v1/jobs` endpoints won't work |
| **Dataset lifecycle states** | No deprecated/deleted state model | Objects removed from registry purge their lineage; no "deprecated" state |

> **🗺️ DATA STRATEGY**: The Feast consumer closes the biggest gap (self-hosted lineage without Marquez) but is not a full Marquez replacement. For RHOAI 3.6, the TP scope (RHAISTRAT-2335) covers the Feast consumer with RBAC hardening and operator integration. Full column-level and version-history capabilities remain a future investment.

## Part 7: The E2E Lineage Vision — Current State

### What Works Today (September 2026)

```
Raw Data (PostgreSQL/S3)
    │
    │  [OpenLineage Spark listener — auto-emits to Feast consumer ✅]
    ▼
ETL / Transformation  (Spark)
    │  ↕ SymlinksDatasetFacet connects Spark output to Feast DataSource
    │
    │  [feast apply — auto-emits + auto-ingests ✅]
    ▼
Feast DataSource → FeatureView → FeatureService
    │
    │  [feast materialize — START/COMPLETE/FAIL ✅]
    ▼
Online Store  ←──────────────────────────────────────────────────────────┐
    │                                                                     │
    │  [No OL events from KFP yet — manual client required ⚠️]           │
    ▼                                                                     │
Model Training (KFP)                                                      │
    │                                                                     │
    │  [MLflow does NOT emit OL events yet ❌ — issue open upstream]      │
    ▼                                                                     │
Model (MLflow Registry)                                                   │
    │                                                                     │
    │  [No serving lineage ❌ — future RFC]                               │
    ▼                                                                     │
Model Inference / Agent Decision                                          │
                                                                          │
All ✅ portions visible in Feast UI Lineage tab ───────────────────────── ┘
```

### Deploying the Cross-Producer Pipeline

```bash
# 1. Start Feast with consumer enabled
feast ui   # or: feast serve_lineage --port 6580

# 2. Configure Spark to send to Feast
spark-submit \
  --conf spark.openlineage.transport.url=http://feast-server:8888 \
  --conf spark.openlineage.transport.endpoint=api/v1/lineage \
  --conf spark.openlineage.namespace=spark://ml-cluster \
  etl_job.py

# 3. Configure Airflow
export OPENLINEAGE_URL=http://feast-server:8888
export OPENLINEAGE_NAMESPACE=airflow://prod
airflow dags trigger credit_pipeline

# 4. Feast operations emit automatically
feast apply          # emits + auto-ingests
feast materialize    # emits START/COMPLETE/FAIL

# 5. View the full graph
open http://localhost:8888  # Feast UI → Lineage tab
```

In [ ]:
# Manual OpenLineage emission — for tools that don't auto-instrument
# (e.g. Ray Data jobs, custom Python pipelines, model training code)
#
# Point the client at the Feast consumer endpoint instead of Marquez.

import os
from openlineage.client import OpenLineageClient
from openlineage.client.run import RunEvent, RunState, Run, Job
from openlineage.client.event_v2 import (
    RunEvent as RunEventV2,
    InputDataset,
    OutputDataset,
)
from openlineage.client.transport.http import HttpConfig, HttpTransport
import uuid

# Point at Feast consumer (instead of Marquez)
FEAST_URL = "http://feast-server:8888"
OL_NAMESPACE = "ray://ml-cluster"

print("Example: Manual OpenLineage emission targeting Feast consumer")
print()
print("# Configure the OL client to send to Feast")
print(f"transport = HttpConfig(url='{FEAST_URL}', endpoint='api/v1/lineage')")
print("client = OpenLineageClient(transport=HttpTransport(transport))")
print()
print("# Or via environment variable (simplest):")
print(f"# export OPENLINEAGE_URL={FEAST_URL}")
print("# from openlineage.client import OpenLineageClient")
print("# client = OpenLineageClient()  # auto-reads OPENLINEAGE_URL")
print()
print("# Emit at job start")
print("run_id = str(uuid.uuid4())")
print("client.emit(RunEvent(")
print("    eventType=RunState.START,")
print(f"    run=Run(runId=run_id),")
print(f"    job=Job(namespace='{OL_NAMESPACE}', name='ray.training.credit_model'),")
print("    inputs=[InputDataset(namespace='lineage_demo', name='lineage_demo.credit_history')],")
print("    outputs=[],")
print("))")
print()
print("# ... do training ...")
print()
print("# Emit at completion")
print("client.emit(RunEvent(")
print("    eventType=RunState.COMPLETE,")
print(f"    run=Run(runId=run_id),")
print(f"    job=Job(namespace='{OL_NAMESPACE}', name='ray.training.credit_model'),")
print("    inputs=[InputDataset(namespace='lineage_demo', name='lineage_demo.credit_history')],")
print("    outputs=[OutputDataset(namespace='mlflow://ml-registry', name='credit_model/v1')],")
print("))")
print()
print("The Feast consumer will then show:")
print("  lineage_demo.credit_history → [Ray Training] → mlflow://ml-registry/credit_model/v1")

## Key Takeaways

### OpenLineage Fundamentals
1. **OpenLineage is the standard** for data lineage — LFAI graduate project, adopted by Spark, Airflow, dbt, Flink and more
2. **Event types**: RunEvent (job execution with inputs/outputs), DatasetEvent, JobEvent
3. **Facets** carry typed metadata extensions — Feast adds its own (FeastFeatureViewFacet, FeastMaterializationFacet, etc.)

### Feast as Producer (unchanged)
4. **Feast auto-emits at `feast apply`** — full dependency graph: DataSource + Entity → FeatureView → FeatureService → SavedDataset
5. **Feast auto-emits at `feast materialize`** — START/COMPLETE/FAIL RunEvents with FeastMaterializationFacet
6. **Registry API/gRPC changes also emit** — not just CLI apply (added Aug 2026)
7. **Config migration**: old `lineage.backend_url` → new `openlineage.transport_url`

### Feast as Consumer (new in June 2026)
8. **Feast is now also a consumer** — enables `POST /api/v1/lineage` to receive events from Spark, Airflow, dbt, KFP
9. **SQL registry required** — consumer stores events in 7 lineage tables in the registry DB
10. **`feast serve_lineage`** — standalone lineage server for independent scaling
11. **Feast UI Lineage tab** — unified graph view of Feast objects + external producer events
12. **SymlinksDatasetFacet** is the key for connecting Spark output datasets to Feast DataSources

### Remaining Gaps vs Marquez
13. **No column-level lineage queries** — facets stored but not indexed
14. **No dataset versioning** — current state only, no version history
15. **No point-in-time graph** — Marquez has this, Feast does not
16. **MLflow still doesn't emit OL events** — breaks the chain at model training

### RHOAI Context
17. **RHAISTRAT-2335** — Feast consumer Tech Preview in RHOAI 3.6, status: Release Pending
18. **Workshop Decision #1** — E2E lineage is the highest-priority data strategy investment
19. **No Marquez required** for Feast-centric lineage — but Marquez still needed for column-level lineage and dataset versioning

## What's Next

- **Module 08**: RAG & Vector Search — embedding features with lineage tracking
- **Module 11**: KFP Integration — pipeline-level lineage (KFP → Feast consumer)
- **Module 13**: UI & Observability — Feast UI lineage tab in depth